## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [2]:
effect_size = sms.proportion_effectsize(0.2, 0.19)

In [3]:
effect_size

np.float64(0.025241594409087353)

In [4]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )

required_n = ceil(required_n)
print(f'Нам потрібно {required_n} користувачів для кожної групи.')
print(f'Сумарно {required_n * 2} користувачів нам потрібно, щоб провести A/B тест.')

Нам потрібно 24638 користувачів для кожної групи.
Сумарно 49276 користувачів нам потрібно, щоб провести A/B тест.


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = pd.read_csv('drive/MyDrive/Date_Analysis/data/cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [7]:
# Впевнимось, що немає користувачів, які були обрані кілька разів:
user_counts = df['userid'].value_counts(ascending=False)
multi_users = user_counts[user_counts>1].count()

print ('Чи є у вибірці користувачі, які були обрані більше одного разу?')
if multi_users == 0:
  print ('Таких користувачів не знайдено.')
else:
  print(f'Так. Знайдено {multi_users} користувачі.')

Чи є у вибірці користувачі, які були обрані більше одного разу?
Таких користувачів не знайдено.


In [8]:
# Середнє значення показника `retention_7`
avg_30 = df[df.version == 'gate_30'].retention_7.mean()
avg_40 = df[df.version == 'gate_40'].retention_7.mean()
print('Показник утримання користувачів на 7 день:')
print(f'Контрольна група: {avg_30:.3f}')
print(f'Тестова група: {avg_40:.3f}')

Показник утримання користувачів на 7 день:
Контрольна група: 0.190
Тестова група: 0.182


Судячи з показників, можемо сформулювати гіпотезу, що версія гри, де ворота розміщені на рівні 30, дає нам краще утримання користувачів, ніж нова версія, з воротами на 40му рівні.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [9]:
print (f'Маємо вибірку з {df.shape[0]} записів.')

Маємо вибірку з 90189 записів.


In [10]:
# Самплінг:
gate30_version = df[df['version'] == 'gate_30'].sample(n=required_n, random_state=22)
gate40_version = df[df['version'] == 'gate_40'].sample(n=required_n, random_state=22)

ab_test = pd.concat([gate30_version, gate40_version], axis=0)
ab_test.reset_index(drop=True, inplace=True)

In [11]:
ab_test.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,7540471,gate_30,45,True,False
1,3589138,gate_30,21,True,False
2,3177668,gate_30,14,True,False
3,2133884,gate_30,26,False,False
4,492763,gate_30,39,True,True


In [12]:
ab_test['version'].value_counts()

,count
version,
gate_30,24638
gate_40,24638


In [13]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [14]:
gate30_results = ab_test[ab_test['version'] == 'gate_30']['retention_7']
gate40_results = ab_test[ab_test['version'] == 'gate_40']['retention_7']

In [15]:
successes = [gate30_results.sum(), gate40_results.sum()]
nobs = [gate30_results.count(), gate40_results.count()]

In [16]:
z_stat, pval = proportions_ztest(successes, nobs)
(lower_30, lower_40), (upper_30, upper_40) = proportion_confint(successes, nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.5f}')
print(f'Довірчий інтервал 95% для версії гри gate_30: [{lower_30:.3f}, {upper_30:.3f}]')
print(f'Довірчий інтервал 95% для версії гри gate_40: [{lower_40:.3f}, {upper_40:.3f}]')

z statistic: 3.20
p-value: 0.00137
Довірчий інтервал 95% для версії гри gate_30: [0.184, 0.194]
Довірчий інтервал 95% для версії гри gate_40: [0.173, 0.183]


### Висновок:

1. Так як, p-value < alpha -  є статистично значуща різниця між поведінкою користувачів у контрольній та тестовій версіях гри.
2. Довірчі інтервали утримання користувачів двох версій гри не перетинаються, і це означає, що є деяка різниця в утриманні користувачів при гейті на 30му рівні та гейті на 40му рівні.

Відсоток утримання для контрольної групи знаходиться в діапазоні 18.4% - 19.4%, як і заявлено продакт менеджером (19% базове утримання). Тестова група показала гірший результат 17.3% - 18.3%. Що не відповідає нашим очікуванням (20%).

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Гіпотези:

$H_0$: між показником утримання користувачів та версією гри немає залежності;

$H_1$: є залежність між показником утримання користувачів та версією гри.

In [17]:
ab_test.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,7540471,gate_30,45,True,False
1,3589138,gate_30,21,True,False
2,3177668,gate_30,14,True,False
3,2133884,gate_30,26,False,False
4,492763,gate_30,39,True,True


In [18]:
crosstab = pd.crosstab(ab_test['version'], ab_test['retention_7'])
crosstab

retention_7,False,True
version,,
gate_30,19978,4660
gate_40,20253,4385


In [19]:
chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 10.166
p-value = 0.00143
Ступені свободи = 1
Очікувані частоти:
 [[20115.5  4522.5]
 [20115.5  4522.5]]


### Висновок:
Отримали p-value = 0.00143 значно менше за alpha = 0.05, тобто, є залежність між версією гри та показником утримання користувача на 7й день після встановлення гри.